# Persiapan Data: Label AQI dan Fitur (PySpark + HDFS)
**Kelompok 07 | Mahadata**

Input : `hdfs://namenode:9000/openaq/raw/GabunganDataset_BigData_eda.csv` (keluaran notebook EDA, satu baris per lokasi per jam)
Output: Parquet `hdfs://namenode:9000/openaq/clean/aqi_fitur` (berisi label AQI dan fitur)

Jalankan sel dari atas ke bawah. Komentar "Nilai acuan" di tiap sel dipakai untuk memeriksa hasil.

## Setup

In [1]:
import numpy as np
from pyspark.sql import SparkSession, functions as F, Window

spark = (SparkSession.builder
         .appName("AQI_PersiapanData")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

EDA   = "hdfs://namenode:9000/openaq/raw/GabunganDataset_BigData_eda.csv"
CLEAN = "hdfs://namenode:9000/openaq/clean/aqi_fitur"
print("Spark", spark.version)

Spark 4.2.1


## 1. Baca CSV hasil EDA
Data sudah dibersihkan di notebook EDA: format wide, jam yang hilang tetap ada sebagai baris kosong, dan PM2.5 bernilai 0 sudah menjadi kosong.

In [2]:
data = (spark.read.csv(EDA, header=True, inferSchema=True)
             .withColumn("ts", F.to_timestamp("datetime_from_utc"))
             .drop("datetime_from_utc")
             .cache())
data.printSchema()
print("jumlah baris:", data.count()) 
data.groupBy("location_name").count().show()
print("PM2.5 kosong:", data.filter(F.col("pm25").isNull()).count())

# Jarak antar baris harus tepat 1 jam supaya lag 1 baris berarti 1 jam sebelumnya
w = Window.partitionBy("location_name").orderBy("ts")
selisih = data.withColumn("selisih_jam", (F.unix_timestamp("ts") - F.unix_timestamp(F.lag("ts").over(w))) / 3600)
print("baris dengan jarak bukan 1 jam:", selisih.filter(F.col("selisih_jam") != 1).count())

root
 |-- location_name: string (nullable = true)
 |-- pm1: double (nullable = true)
 |-- pm25: double (nullable = true)
 |-- relativehumidity: double (nullable = true)
 |-- temperature: double (nullable = true)
 |-- um003: double (nullable = true)
 |-- ts: timestamp (nullable = true)

jumlah baris: 14406
+--------------------+-----+
|       location_name|count|
+--------------------+-----+
|          Tambulaung| 7203|
|Taman Tun Dr. Ismail| 7203|
+--------------------+-----+

PM2.5 kosong: 763
baris dengan jarak bukan 1 jam: 0


## 2. Label AQI (PM2.5 jam t, EPA versi terbaru)
0 Baik (<=9,0) | 1 Sedang (<=35,4) | 2 Tidak sehat bagi kelompok sensitif (<=55,4) | 3 Tidak sehat (<=125,4) | 4 Sangat tidak sehat (<=225,4) | 5 Berbahaya

In [3]:
data = data.withColumn(
    "label",
    F.when(F.col("pm25").isNull(), F.lit(None).cast("double"))
     .when(F.col("pm25") <= 9.0, 0.0)
     .when(F.col("pm25") <= 35.4, 1.0)
     .when(F.col("pm25") <= 55.4, 2.0)
     .when(F.col("pm25") <= 125.4, 3.0)
     .when(F.col("pm25") <= 225.4, 4.0)
     .otherwise(5.0))

data.groupBy("location_name", "label").count().orderBy("location_name", "label").show(20)

+--------------------+-----+-----+
|       location_name|label|count|
+--------------------+-----+-----+
|Taman Tun Dr. Ismail| NULL|   23|
|Taman Tun Dr. Ismail|  0.0|  271|
|Taman Tun Dr. Ismail|  1.0| 3080|
|Taman Tun Dr. Ismail|  2.0| 2654|
|Taman Tun Dr. Ismail|  3.0| 1003|
|Taman Tun Dr. Ismail|  4.0|  172|
|          Tambulaung| NULL|  740|
|          Tambulaung|  0.0| 3566|
|          Tambulaung|  1.0| 2159|
|          Tambulaung|  2.0|  559|
|          Tambulaung|  3.0|  140|
|          Tambulaung|  4.0|   39|
+--------------------+-----+-----+



## 3. Fitur
PM2.5 jam yang sama **tidak** dipakai sebagai fitur (mencegah data leakage). Semua pengukuran sensor memakai nilai jam sebelumnya.

In [4]:
w = Window.partitionBy("location_name").orderBy("ts")

for k in (1, 2, 3):
    data = data.withColumn(f"pm25_lag{k}", F.lag("pm25", k).over(w))
data = data.withColumn("pm25_ma3", F.avg("pm25").over(w.rowsBetween(-3, -1)))

for c in ["pm1", "um003", "temperature", "relativehumidity"]:
    data = data.withColumn(f"{c}_lag1", F.lag(c, 1).over(w))

data = (data
        .withColumn("ts_local", F.expr("ts + INTERVAL 8 HOURS"))      # Malaysia UTC+8
        .withColumn("jam", F.hour("ts_local"))
        .withColumn("jam_sin", F.sin(2 * np.pi * F.col("jam") / 24))
        .withColumn("jam_cos", F.cos(2 * np.pi * F.col("jam") / 24))
        .withColumn("is_ttdi", (F.col("location_name") == "Taman Tun Dr. Ismail").cast("double")))

FITUR = ["pm25_lag1", "pm25_lag2", "pm25_lag3", "pm25_ma3",
         "pm1_lag1", "um003_lag1", "temperature_lag1", "relativehumidity_lag1",
         "jam_sin", "jam_cos", "is_ttdi"]

final = (data
         .dropna(subset=FITUR + ["label"])
         .select("location_name", "ts", "ts_local", "pm25", "label", *FITUR))

print("baris akhir (siap model):", final.count())
final.groupBy("location_name").count().show()
final.show(5)

baris akhir (siap model): 13135
+--------------------+-----+
|       location_name|count|
+--------------------+-----+
|Taman Tun Dr. Ismail| 7144|
|          Tambulaung| 5991|
+--------------------+-----+

+--------------------+-------------------+-------------------+----+-----+---------+---------+---------+------------------+--------+----------+----------------+---------------------+--------------------+-------------------+-------+
|       location_name|                 ts|           ts_local|pm25|label|pm25_lag1|pm25_lag2|pm25_lag3|          pm25_ma3|pm1_lag1|um003_lag1|temperature_lag1|relativehumidity_lag1|             jam_sin|            jam_cos|is_ttdi|
+--------------------+-------------------+-------------------+----+-----+---------+---------+---------+------------------+--------+----------+----------------+---------------------+--------------------+-------------------+-------+
|Taman Tun Dr. Ismail|2026-01-01 03:00:00|2026-01-01 11:00:00|36.8|  2.0|     37.0|     36.1|     38

## 4. Simpan ke HDFS sebagai Parquet

In [5]:
final.write.mode("overwrite").parquet(CLEAN)

cek = spark.read.parquet(CLEAN)
print("baca ulang dari Parquet:", cek.count(), "baris,", len(cek.columns), "kolom")
cek.printSchema()

baca ulang dari Parquet: 13135 baris, 16 kolom
root
 |-- location_name: string (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- ts_local: timestamp (nullable = true)
 |-- pm25: double (nullable = true)
 |-- label: double (nullable = true)
 |-- pm25_lag1: double (nullable = true)
 |-- pm25_lag2: double (nullable = true)
 |-- pm25_lag3: double (nullable = true)
 |-- pm25_ma3: double (nullable = true)
 |-- pm1_lag1: double (nullable = true)
 |-- um003_lag1: double (nullable = true)
 |-- temperature_lag1: double (nullable = true)
 |-- relativehumidity_lag1: double (nullable = true)
 |-- jam_sin: double (nullable = true)
 |-- jam_cos: double (nullable = true)
 |-- is_ttdi: double (nullable = true)



## 5. Cek akhir: distribusi kelas latih (Jan-Agu) dan uji (Sep-Okt)

In [6]:
batas = F.lit("2026-09-01 00:00:00").cast("timestamp")
(cek.withColumn("bagian", F.when(F.col("ts_local") < batas, "latih").otherwise("uji"))
    .groupBy("location_name", "bagian", "label").count()
    .orderBy("location_name", "bagian", "label")
    .show(40))

+--------------------+------+-----+-----+
|       location_name|bagian|label|count|
+--------------------+------+-----+-----+
|Taman Tun Dr. Ismail| latih|  0.0|  244|
|Taman Tun Dr. Ismail| latih|  1.0| 2905|
|Taman Tun Dr. Ismail| latih|  2.0| 2255|
|Taman Tun Dr. Ismail| latih|  3.0|  354|
|Taman Tun Dr. Ismail| latih|  4.0|    7|
|Taman Tun Dr. Ismail|   uji|  0.0|   26|
|Taman Tun Dr. Ismail|   uji|  1.0|  158|
|Taman Tun Dr. Ismail|   uji|  2.0|  381|
|Taman Tun Dr. Ismail|   uji|  3.0|  649|
|Taman Tun Dr. Ismail|   uji|  4.0|  165|
|          Tambulaung| latih|  0.0| 2836|
|          Tambulaung| latih|  1.0| 1587|
|          Tambulaung| latih|  2.0|  247|
|          Tambulaung| latih|  3.0|   47|
|          Tambulaung| latih|  4.0|    5|
|          Tambulaung|   uji|  0.0|  286|
|          Tambulaung|   uji|  1.0|  549|
|          Tambulaung|   uji|  2.0|  307|
|          Tambulaung|   uji|  3.0|   93|
|          Tambulaung|   uji|  4.0|   34|
+--------------------+------+-----